# PrecisionSkin - Monk Skin Tone (MST) classifier (separate research model)

**What this is.** A skin-tone model on the 10-shade Monk Skin Tone scale, trained on **MST-E** (reference examples) and **SCIN** (supplementary, tone labels are layperson *estimates*).
It is **independent** of the MobileNetV2 condition classifier and of the lesion detector: its output never changes a condition result.

**Why it is built this way** (lesson from the retired Fitzpatrick model, whose "labels" were folders that encoded ethnicity):
1. every split keeps all photos of one person / SCIN case together (group split);
2. labels come from raters, never from another dataset's folder names;
3. no hue / saturation augmentation;
4. results are reported as exact shade, **within 1 shade** and **3 bands**, with a range from resampling people, plus a brightness stress test.

**Run order:** cell 1 (settings) -> 2 (audit) -> 3 (manifest + split) -> 4 (train) -> 5 (evaluate). Kernel: the `.venv` one with TensorFlow ("PrecisionSkinV1 GPU").
**Before training:** read the licence / terms of MST-E and SCIN for model training (not only reference use).

In [ ]:
# CELL 1 - settings
import os, sys, json
from pathlib import Path
sys.path.insert(0, "/mnt/c/Users/Bautista/Downloads")          # mst_skintone.py lives here (also in the repo)
import mst_skintone as M

SCIN_DIR = Path("/mnt/c/Users/Bautista/Downloads/MST_Data/SCIN")     # folder with scin_cases.csv, scin_labels.csv and the images (dataset/images/...)
MSTE_DIR = Path("/mnt/c/Users/Bautista/Downloads/MST_Data/MST-E")    # folder with the MST-E images (an MST number in the file / folder name)
OUT      = Path.home() / "PrecisionSkinV1" / "skintone_mst"           # everything this notebook writes (on the WSL disk)
FACE_ONLY = True        # SCIN: keep only cases flagged head / neck (many SCIN photos are body parts)
USE_SCIN  = True
MAX_PER_CASE = 1        # SCIN: photos of one case are near-duplicates, take one
SEED = 42
EPOCHS_HEAD, EPOCHS_FT, BATCH = 6, 14, 32
OUT.mkdir(parents=True, exist_ok=True)
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
print("SCIN:", SCIN_DIR, SCIN_DIR.exists(), "| MST-E:", MSTE_DIR, MSTE_DIR.exists())

In [ ]:
# CELL 2 - audit the data you downloaded (nothing is written)
if SCIN_DIR.exists():
    scin_df, scin_cols = M.audit_scin(SCIN_DIR)
else:
    print("SCIN folder not found - download it first (see the folder names above).")
print()
if MSTE_DIR.exists():
    M.audit_mste(MSTE_DIR)
else:
    print("MST-E folder not found - download it first.")
print("\nCheck: the Monk column was detected, most cases have a label, the head/neck flag was found, MST-E shows 19-ish subjects.")

In [ ]:
# CELL 3 - one table of photos (path, MST, person/case group, source) and a person-based split
import pandas as pd
df = M.build_manifest(SCIN_DIR if SCIN_DIR.exists() else None, MSTE_DIR if MSTE_DIR.exists() else None, face_only=FACE_ONLY, use_scin=USE_SCIN, max_per_case=MAX_PER_CASE)
assert len(df) > 0, "No photos found - check the folders and the MST number in the MST-E names"
df = M.group_split(df, seed=SEED)
df.to_csv(OUT / "manifest_split.csv", index=False)
print("photos:", len(df), "| people/cases:", df.group.nunique())
print(pd.crosstab(df.mst, df.split).reindex(range(1, 11), fill_value=0).to_string())
print("\nby source:", df.groupby(["source", "split"]).size().unstack(fill_value=0).to_dict("index"))
empty = [b for b in ("light", "medium", "dark") if (df[(df.split == "test")].band == b).sum() < 10]
print("WARNING: fewer than 10 test photos in band(s):", empty, "- results for those bands are not reliable" if empty else "\nEvery band has at least 10 test photos.")
print("Shades with no training photo:", [k for k in range(1, 11) if (df[(df.split == "train")].mst == k).sum() == 0])

In [ ]:
# CELL 4 - train (head first, then the top MobileNetV2 layers)
model = M.train_model(df, OUT, epochs_head=EPOCHS_HEAD, epochs_ft=EPOCHS_FT, batch=BATCH, seed=SEED)
model.save(OUT / "mst_final.keras")
print("saved:", OUT / "mst_final.keras")

In [ ]:
# CELL 5 - honest evaluation on photos of people the model has never seen
import numpy as np
test = df[df.split == "test"].reset_index(drop=True)
m = M.evaluate(model, test)
print("\n".join(M.verdict(m)))
print("\nby source (within 1 shade):", {k: round(v["within1"], 3) for k, v in m["by_source"].items()})
print("brightness stress (within 1):", {k: round(v["within1"], 3) for k, v in m["brightness_stress"].items()})
conf = np.array(m["confusion"]); print("\nconfusion (rows = true MST 1..10, columns = predicted):\n", conf)
json.dump({k: v for k, v in m.items()}, open(OUT / "mst_results.json", "w"), indent=2)
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4.5)); ax.imshow(conf, cmap="Blues"); ax.set_xticks(range(10)); ax.set_xticklabels(range(1, 11)); ax.set_yticks(range(10)); ax.set_yticklabels(range(1, 11))
    ax.set_xlabel("predicted MST"); ax.set_ylabel("true MST"); fig.tight_layout(); fig.savefig(OUT / "mst_confusion.png", dpi=130); print("saved", OUT / "mst_confusion.png")
except Exception as ex:
    print("plot skipped:", ex)
print("\nSaved to:", OUT)
print("Limits to state in the paper: SCIN tone labels are layperson estimates; MST-E has few people; the model has not been checked on kiosk (ring-light) captures yet.")